# Spatial Relationships


In the previous section, we learned how to modify feature geometry and create new spatial datasets. However, many tasks require not just transforming geometry but also understanding **how features are positioned relative to one another**.

Spatial relationships allow us to determine whether features intersect, whether one lies inside another, or whether they share a boundary. These operations underpin many common spatial analysis tasks, such as:

- finding features within a given area;
- assessing accessibility;
- filtering data by location.

In GeoPandas, this is handled through methods known as **spatial predicates** — for example: `intersects`, `within`, `contains`.

In this section, we will look at how to apply spatial predicates to analyse and filter geodata.


## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import pandas as pd
import geopandas as gpd
import osmnx as ox

# cache OSM responses on disk, so repeating a query does not hit the server again
ox.settings.cache_folder = "../../cache"

### 0.2. Preparing the Data


We start with the boundary of the Innere Stadt — the first district of Vienna and its historic core — loaded from OpenStreetMap.


In [ ]:
area_name = "Innere Stadt, Vienna, Austria"
admin_border = ox.geocode_to_gdf(area_name)

admin_border.explore(tiles="cartodbpositron")

Then a set of well-known places to visit across Vienna — museums, cafés, concert halls, shops — read from a CSV file and turned into a `GeoDataFrame`.

The file is **vienna_top_locations.csv** from `data/vienna/`. _Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._


In [ ]:
locations_csv = pd.read_csv("../../data/vienna/vienna_top_locations.csv", sep=";", decimal=",")
locations_csv = locations_csv.dropna(subset=["geo_longitude", "geo_latitude"])

locations_gdf = gpd.GeoDataFrame(
    locations_csv,
    geometry=gpd.points_from_xy(locations_csv["geo_longitude"], locations_csv["geo_latitude"]),
    crs="EPSG:4326"
)

locations_gdf.explore(tiles="cartodbpositron")

## 1. Spatial Predicates

**Spatial predicates** describe the geometric relationship between two spatial features (points, lines, polygons, etc.).

They answer questions such as:
"Do these features intersect?",
"Is one feature inside another?",
"Do their boundaries touch?" and so on.

Spatial predicates form the foundation of spatial queries, spatial joins, data filtering, and proximity analysis.

### 1.1. Common Spatial Predicates

The table below describes the main binary spatial relationships between geometries:

| **Predicate**   | **Description**                                                                                                                                                                          |
| --------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **intersects**  | **Share at least one point** — the geometries have at least one point in common (whether on their boundaries or interiors). Equivalent to the negation of `disjoint`.                    |
| **disjoint**    | **Share no points** — the geometries are completely separate and have no points in common.                                                                                               |
| **within**      | **A is inside B** — geometry A lies entirely within geometry B; all points of A are in the interior of B. Equivalent to `B.contains(A)`.                                                 |
| **contains**    | **A contains B** — geometry A fully contains geometry B, with at least one point of B lying strictly in the interior of A. Equivalent to `B.within(A)`.                                 |
| **touches**     | **Boundaries meet** — the geometries share points only on their boundaries; their interiors do not intersect.                                                                            |
| **overlaps**    | **Partial overlap** — the geometries partially share an area but neither fully contains the other (typically applies to features of the same dimension, e.g. polygon–polygon).           |
| **crosses**     | **Intersection with dimension reduction** — the interiors of the geometries intersect, and the result has a lower dimension (e.g. a line crossing a polygon, or two lines meeting at a point). |


### 1.2. Spatial Predicates in GeoPandas

In GeoPandas, spatial predicates are implemented as methods applied to geometries that return boolean values (`True` or `False`).

Each method checks whether a given spatial relationship holds for each pair of geometries — for example, `within`, `contains`, `intersects`, and so on.

In practice, this means you can filter and analyse data based on the spatial relationships between features.

Both layers must be in the same CRS for a predicate to give meaningful results, so let's check that first:

In [ ]:
locations_gdf.crs == admin_border.crs

They match, so we can go ahead. Let's check which locations fall within the Innere Stadt.

In [ ]:
locations_gdf.geometry.within(admin_border.geometry.iloc[0])

The result is a boolean Series where:

- `True` — the feature satisfies the condition (lies within the boundary);
- `False` — it does not.

Now to how spatial predicates are used in practice, to filter data by location.


## 2. Filtering Features by Location


One of the most common tasks in spatial analysis is selecting features that satisfy a given spatial condition — for example, **selecting all points that fall within a polygon**. This is known as a spatial filter, or filtering by location.

Let's filter all locations that fall within the Innere Stadt.


In [ ]:
locations_centre = locations_gdf[
    locations_gdf.geometry.within(admin_border.geometry.iloc[0])
]

print(f"Locations in the city: {len(locations_gdf)}")
print(f"Locations in the district: {len(locations_centre)}")

And the result on a map:

In [ ]:
locations_centre.explore(tiles="cartodbpositron")

The map shows only the locations that fall within the Innere Stadt.


## 3. When the Predicate You Pick Changes the Answer

The filter above used `within`. Had we used `intersects` instead, the result would have been identical — 64 locations either way. That is not a coincidence, and it is worth understanding, because it makes the choice of predicate look unimportant when it is not.

A **point** either falls inside a polygon or it does not. The only case where `within` and `intersects` disagree is a point sitting exactly on the boundary, which almost never happens with real coordinates. For point-in-polygon work, then, the predicates are interchangeable in practice.

**Areas are a different matter.** A polygon can lie fully inside another, or merely overlap its edge — and those are different answers to different questions. Let's compare, using Vienna's census districts against the boundary of the Innere Stadt.

In [ ]:
zaehlbezirke = gpd.read_file("../../data/vienna/vienna_admin.gpkg", layer="zaehlbezirk")

boundary = admin_border.geometry.iloc[0]

print(f"within     : {zaehlbezirke.within(boundary).sum()}")
print(f"intersects : {zaehlbezirke.intersects(boundary).sum()}")

One against twenty-one. `within` counts only the census districts lying entirely inside the Innere Stadt; `intersects` counts every one that touches it at all, including those that merely share a stretch of its border.

Neither number is wrong — they answer different questions. But a report that says "21 census districts in the Innere Stadt" when it meant the first figure is off by a factor of twenty, and nothing in the code will warn you. So the rule is worth stating plainly:

- use **`within`** when you mean *fully inside*;
- use **`intersects`** when you mean *touches at all*;
- and when the features being tested are points, either will do.

## Summary

In this section, we explored **spatial predicates** and learned how to use them to analyse geodata.

We learned:

- what spatial predicates are and what types exist;
- how to use the corresponding methods in GeoPandas;
- how to filter features based on their spatial relationships.

In the examples above, we used spatial predicates to filter data. However, when working with multiple datasets, you often need not just to select features but to combine their attributes.

This is done through a **spatial join**, which we will cover in the next section.
